## ASK YOURSELF
  - What is ONE ROW? (one what?)
  - What is the TARGET? Classification or regression?
  - What DECISION will the prediction support?
  - Which mistake is worse: calling a low earner "high" or missing a high earner?
  - Which columns might NOT exist at the moment of prediction?
  - Which METRICS will I report, and why not only one?


## Stage 1
- Guess 1: number of rows  ~ 48,000
- Guess 2: percent of people earning >50K  ~ 38%
- Guess 3: percent of rows with missing data ~ 20%
- Guess 4: the column that predicts income best ~ fnlwgt

* Blanks
- One row of this file = one person.
- The target column is called income and has 2 possible values, so this is binary classification.
- A model that always answers "<=50K" is a baseline model (name of this reference model). Its score is the line every real model must beat.
- When one class is much rarer than the other, accuracy can look high while the model is useless for the rare class. So I will also report recall and precision (name two metrics from Day 9).
- Prediction-time check: a column that is only known AFTER the outcome is called data leakage.

## 5-line Problem Statement
- One row is one person's data. The target for this data is a person's income(“>50K” vs “<=50K”). The decision is the bank uses the prediction to choose which people to pre-screen first for high-income offer. The worse mistake is missing a real high earner (false negative). The bank loses a good prospect. The opposite mistake, calling a low earner “high”, only wastes a cheap screening, so it costs less. I will protect recall >50k class, because missing a high earner will be the worst mistake. Also i will look precision, so the model does not flag everybody. I report AUC to compare the models without depending on one threshold. I compare accuray against baseline, because accuracy alone can mislead.

## Stage 2
- A file without a header row is read with  header=None  and the column names are given with  names=columns .
- If I forget that, pandas makes the first PERSON become the header and I silently lose 1 row.
- The method that prints each column's type and non-null count is df.info() .
- Text columns have the type str (or string); whole-number columns have the type int64 .
- Number of numeric columns: 6 ; number of text columns: 9 .
- The column "education-num" is a number, but it behaves like a categorical (ordered category). 
* Why does that matter later? It is matter because a straight-line model like logistic regression treats the step from 1 to 2 as equal to the step from 15 to 16. A tree doesn’t care about that. It matches the text column education exactly, with 16 values on each side, so they carry the same information. If i keep both, i double-count it.


In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)


In [2]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/adult-all.csv"
columns = [
    "age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
    "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
    "hours-per-week", "native-country", "income"
]

df = pd.read_csv(url, header=None, names=columns)
df_raw = df.copy()
print(df.shape)
print(df_raw.shape)
mid_idx = len(df) // 2
middle_5_rows = df.iloc[mid_idx - 2 : mid_idx + 3]
print(middle_5_rows)

(48842, 15)
(48842, 15)
       age  workclass  fnlwgt   education  education-num      marital-status  \
24419   35    Private  225860   Assoc-voc             11  Married-civ-spouse   
24420   43    Private  129853        10th              6       Never-married   
24421   50    Private   99925     HS-grad              9  Married-civ-spouse   
24422   58    Private  227800     1st-4th              2           Separated   
24423   55  State-gov  111130  Assoc-acdm             12            Divorced   

              occupation   relationship                race   sex  \
24419       Craft-repair        Husband               White  Male   
24420  Machine-op-inspct  Not-in-family               Black  Male   
24421       Adm-clerical        Husband               White  Male   
24422    Farming-fishing  Not-in-family               Black  Male   
24423       Adm-clerical      Own-child  Asian-Pac-Islander  Male   

       capital-gain  capital-loss  hours-per-week native-country income  
24419 

In [3]:
print(df.dtypes)

age               int64
workclass           str
fnlwgt            int64
education           str
education-num     int64
marital-status      str
occupation          str
relationship        str
race                str
sex                 str
capital-gain      int64
capital-loss      int64
hours-per-week    int64
native-country      str
income              str
dtype: object


In [4]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             48842 non-null  int64
 1   workclass       48842 non-null  str  
 2   fnlwgt          48842 non-null  int64
 3   education       48842 non-null  str  
 4   education-num   48842 non-null  int64
 5   marital-status  48842 non-null  str  
 6   occupation      48842 non-null  str  
 7   relationship    48842 non-null  str  
 8   race            48842 non-null  str  
 9   sex             48842 non-null  str  
 10  capital-gain    48842 non-null  int64
 11  capital-loss    48842 non-null  int64
 12  hours-per-week  48842 non-null  int64
 13  native-country  48842 non-null  str  
 14  income          48842 non-null  str  
dtypes: int64(6), str(9)
memory usage: 5.6 MB
None


## Stage 3
- S3-1  isna() only finds numerical values. Here the missing marker is the
        text "?" , so isna().sum() shows null for every column.
- S3-2  To count the marker per column I compare the whole table with the text
        using df == marker , then call .sum() .
- S3-3  Fill this table from YOUR output:
            column            count of the marker     percent of 100%
            workclass             2799                      5.73
            occupation            2809                      5.75
            native-country         857                      1.75
        Rows with the marker in AT LEAST ONE column: 3620 ( 7.41% )
- S3-4  Two of these columns are missing for almost the same rows. That means the missingness is not random; it is a pattern .
- S3-5  To make the marker a real missing value I can use  df.replace("?", np.nan)
        or  read_csv(url, names=columns nan_values="?")   [NEW option: look it up].
- S3-6  Your three choices for missing categories:
            (a) drop the rows,  (b) fill with the mode (most common value),
            (c) keep "?" as its own value.
        To choose, compare the share of ">50K" among rows WITH the marker
        versus rows WITHOUT it:  with =  13.23%  vs  without = 24.78% .
        DECISION: because the missingness will impact the target class so I choose (c).
- S3-7  Exact duplicate rows found with df.duplicated().sum() : 52 rows.
- S3-8  With no ID column, two different people could look identical, so why is deleting them probably safe HERE? It is safe because fnlwgt represents the estimated number of people in the actual popultion in this specific rows.
- S3-9  Duplicates must be removed BEFORE the split, because otherwise the same row can sit in train AND test and the test score becomes bias .
- S3-10 Suspicious values to inspect: the maximum of capital-gain = 99999 , the maximum of hours-per-week = 99 , the maximum age = 90 . Which of these three is a CODE rather than a real amount? capital-gain

WRITE: DECISION for missing, duplicates and suspicious values (3 short lines).
- Decision for missing is keep the missing value as its own class and for duplicated we
must drop it because if we do the split later without drop the duplicates it will impact
the test sscores and the model become bias. About the suspicious value, I decided to capped the
capital-gain = 99999


In [5]:
marker = "?"
q = (df == marker)
counts = q.sum()
pct = (counts / len(df) * 100).round(2)
table = pd.DataFrame({"count": counts, "percent": pct})
print(table[table["count"] > 0])

                count  percent
workclass        2799     5.73
occupation       2809     5.75
native-country    857     1.75


In [6]:
rows_any = q.any(axis=1)
print(rows_any.sum(), round(rows_any.mean() * 100, 2))

3620 7.41


In [7]:
print(pd.crosstab(q["workclass"], q["occupation"]))

occupation  False  True 
workclass               
False       46033     10
True            0   2799


In [8]:
target01 = (df["income"] == ">50K").astype(int)
print((target01.groupby(rows_any).mean() * 100).round(2))

False    24.78
True     13.23
Name: income, dtype: float64


In [9]:
print(df.duplicated().sum())
df_nodup = df.drop_duplicates()
print(df.shape, df_nodup.shape)
print(df["fnlwgt"].nunique())

52
(48842, 15) (48790, 15)
28523


In [10]:
print(df[["age", "hours-per-week", "capital-gain"]].agg(["min", "max"]))
print((df["capital-gain"] == df["capital-gain"].max()).sum())

     age  hours-per-week  capital-gain
min   17               1             0
max   90              99         99999
244


In [11]:
df_clean = df.replace("?", np.nan)
print(df_clean.isna().sum()[lambda s: s > 0])

workclass         2799
occupation        2809
native-country     857
dtype: int64


## Stage 4
- S4-1  value_counts(normalize=True) gives proportions instead of counts.
- S4-2  capital-gain is exactly 0 for 91.7 % of people. Its median is 0.0 but its
      mean is 1079.07 , so the distribution is right-skewed with a spike at 0 .
- S4-3  Among people with capital-gain > 0, 61.73 % earn >50K; among those with
      capital-gain = 0, 20.52 % do. A column like this is useful even though it
      is mostly zero because when it is non-zero, it is a highly predictive, strong positive indicator for the target class (>50K).
- S4-4  "education" has 16 unique values and "education-num" has 16 . The pair
      table shows each education matches exactly one number, so they carry
      the same information. DECISION: keep education-num , drop education.
- S4-5  crosstab of relationship x sex: the group "Husband" is 100 % male. So
      "relationship" is a proxy for sex.
- S4-6  native-country: 89.7 % of rows are "United-States". With 42 countries,
      one-hot encoding would create many columns that are almost always 0 .
      DECISION: group the 41 rare countries together into a single category labeled "Other" or apply target encoding.
- S4-7  fnlwgt is a population sampling weight, not a property of the person. Its correlation
      with the 0/1 target is about -0.006 . DECISION: keep or drop? drop .
- S4-8  df.groupby("column")["target01"].mean() gives the rate of ">50K" per
      group (the method that averages a 0/1 column).
- S4-9  Top-3 groups by rate (from your output): Exec-Managerial , Prof-speciality , Protective-serv .
      Bottom group: Priv-house-serv .
- S4-10 A correlation of 0.33 between education-num and income means
      "they move together a little", NOT causation .
- S4-11 Correlation with a 0/1 target only measures linear (straight-line)
      relationships. A column can matter a lot and still show a small

In [12]:
num_cols = ["age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
print(df[num_cols].describe().loc[["min", "50%", "mean", "max"]].round(1))

       age     fnlwgt  education-num  capital-gain  capital-loss  \
min   17.0    12285.0            1.0           0.0           0.0   
50%   37.0   178144.5           10.0           0.0           0.0   
mean  38.6   189664.1           10.1        1079.1          87.5   
max   90.0  1490400.0           16.0       99999.0        4356.0   

      hours-per-week  
min              1.0  
50%             40.0  
mean            40.4  
max             99.0  


In [13]:
print(df["native-country"].value_counts(normalize=True).head(3).round(3))

native-country
United-States    0.897
Mexico           0.019
?                0.018
Name: proportion, dtype: float64


In [14]:
g = df["capital-gain"]
print((g == 0).mean() * 100)
print(g.median(), g.mean(), g.skew())


91.73866754023177
0.0 1079.0676262233324 11.894658996587946


In [15]:
print((target01.groupby(df["capital-gain"] == 0).mean() * 100).round(1))

capital-gain
False    61.7
True     20.5
Name: income, dtype: float64


In [16]:
print(df["education"].nunique(), df["education-num"].nunique())
print(df[["education", "education-num"]].drop_duplicates().shape[0])

16 16
16


In [17]:
print(pd.crosstab(df["relationship"], df["sex"], normalize="index").round(3))

sex             Female   Male
relationship                 
Husband          0.000  1.000
Not-in-family    0.467  0.533
Other-relative   0.458  0.542
Own-child        0.445  0.555
Unmarried        0.766  0.234
Wife             0.999  0.001


In [18]:
print(df["fnlwgt"].corr(target01).round(4))
print(df["education-num"].corr(target01).round(3), df["age"].corr(target01).round(3))

-0.0063
0.333 0.23


In [19]:
r = target01.groupby(df["occupation"]).agg(["mean", "size"]).sort_values("mean")
print(r.round(3))

                    mean  size
occupation                    
Priv-house-serv    0.012   242
Other-service      0.041  4923
Handlers-cleaners  0.067  2072
?                  0.094  2809
Farming-fishing    0.116  1490
Machine-op-inspct  0.123  3022
Adm-clerical       0.137  5611
Transport-moving   0.204  2355
Craft-repair       0.226  6112
Sales              0.268  5504
Tech-support       0.290  1446
Protective-serv    0.313   983
Armed-Forces       0.333    15
Prof-specialty     0.451  6172
Exec-managerial    0.478  6086


In [20]:
print(df.groupby("income")["age"].describe())

          count       mean        std   min   25%   50%   75%   max
income                                                             
<=50K   37155.0  36.872184  14.104118  17.0  25.0  34.0  46.0  90.0
>50K    11687.0  44.275178  10.558983  19.0  36.0  43.0  51.0  90.0


In [21]:
print(repr(df["income"].iloc[0]))
print(df["income"].value_counts())
target01 = (df["income"] == ">50K").astype(int)
p = target01.mean()
print(round(p * 100, 2), round((1 - p) * 100, 2))
print((target01.groupby(df["sex"]).mean() * 100).round(1))

'<=50K'
income
<=50K    37155
>50K     11687
Name: count, dtype: int64
23.93 76.07
sex
Female    10.9
Male      30.4
Name: income, dtype: float64


In [24]:
has_gain = df["capital-gain"] > 0
no_gain = df["capital-gain"] == 0

print((df[has_gain]["income"] == ">50K").mean() * 100)
print((df[no_gain]["income"] == ">50K").mean() * 100)

61.734820322180916
20.523578905081795


## Stage 5
## Four label
QUESTION : I am trying to determine if education and education-num are identical, 1-to-1 structural duplicates of each other.
CHECK    : I used df[["education", "education-num"]].drop_duplicates().shape[0].
FINDING  : I saw exactly 16 distinct text-to-number pairs, which matches the unique count of each column perfectly.
DECISION : I will drop education because it is a redundant and keep education-num because it carries the exact same information in numeric.
- S5-1  To see the exact text of one value, print it with repr(value) so
        that hidden spaces become visible. [NEW habit]
- S5-2  Target to numbers: ">50K" -> 1 and "<=50K" -> 0 , built with the
    comparison  df["income"] == ">50K"  followed by .astype(int).
- S5-3  Rate of the positive class = 23.93% % . Baseline accuracy of "always
    <=50K" = 76.07 % % . Baseline AUC = 0.50 .
- S5-4  Leakage means a column carries information that is not available
    at prediction time , or that is computed using the target.
- S5-5  Audit table. For every column write L (leaks), OK, or ? (needs thought):
        fnlwgt OK  capital-gain OK   relationship OK   education OK
        hours-per-week OK   marital-status OK   occupation ? : Needs a thought because if an individual is currently unemployed or a student, recording a specific historical or placeholder occupation might introduce a target leakage depending on how the data pipeline handles missing parameters.
    In ONE sentence, explain your answer for the column you marked "?".
- S5-6  In this dataset, I believe the number of columns that truly leak the
    target is 0 because in this dataset the data are collected at the same time, None of the columns or rows updated or added after that. (Be ready to defend it.)
- S5-7  Sensitive columns here: sex and race . Removing "sex" may not
    remove the information because relationship already encodes it.

## Final Decision
- education-num	It is the ordered numeric version of education (16 values, one-to-one), and the strongest numeric link to income (correlation 0.333).
- age	It is known at prediction time. Income tends to rise with career stage (correlation 0.230).
- hours-per-week	It measures work volume. The >50K rate climbs from 6.8% at 20 hours or fewer to 40.6% at 40–60 hours.
- capital-gain Keep, but flagged “?”. The pattern is too sharp to be natural. Gains of 1–3,000 have 0% >50K, gains above 7,000 have 98–100%, and all 244 rows at 99999 are >50K. It may be part of how the label was built, and the data alone can’t settle that. I will run the model with and without it in Stages 8–9 and report both scores.
- capital-loss	It is a weak signal (solo AUC 0.534) and 95.3% of values are zero. It is harmless to keep and known at prediction time.
- marital-status	Solo AUC is 0.770. Married-civ-spouse earns >50K 44.6% of the time against 4.5% for Never-married. It overlaps with relationship.
- relationship (suggested)	Keep, flagged as a proxy for sex. It has the strongest solo AUC (0.780). Husband is 99.99% male and Wife is 99.87% female. The cost is that gender information stays in the model, so I check results by group in Stage 10.
- workclass	Solo AUC is 0.585. The “?” rows are kept as their own level, because they have a 13.23% positive rate against 24.78% for the rest.
- occupation	Solo AUC is 0.73. Exec-managerial has a 47.8% rate and Priv-house-serv has 1.2%. The “?” rows are kept as their own level.
- native-country Keep as a weak signal. 89.7% of rows are US, and the non-US rate is 19.8% against 24.4% for the US. It is one-hot encoded with “?” as a level. Its solo AUC is only 0.529, so it adds little, but it costs nothing to keep.
- education	It is a text duplicate of education-num (16 values each, one-to-one), so it adds nothing new.
- fnlwgt	It is a survey sampling weight, not a trait of the person. It has no predictive value (solo AUC 0.504, correlation -0.006).
- sex	It is a sensitive attribute, so I remove it to stop the model using it directly. Limit: relationship and marital-status still carry most of that information, and a forest without sex scored about the same (0.912). I am removing the input, not the information.
- race	It is a sensitive attribute with weak predictive value (solo AUC 0.538). Dropping the column does not guarantee fair outcomes, because other columns such as native-country can act as proxies. I will check results by group in Stage 10.

In [28]:
from sklearn.metrics import roc_auc_score

rows = []
for col in df.columns.drop("income"):
    s = df[col]
    if pd.api.types.is_numeric_dtype(s):
        score = s
    else:
        score = s.map(target01.groupby(s).mean())
    auc = roc_auc_score(target01, score)
    rows.append((col, round(max(auc, 1 - auc), 3)))
audit = pd.DataFrame(rows, columns=["column", "solo_auc"]).sort_values("solo_auc", ascending=False)
print(audit.to_string(index=False))

        column  solo_auc
  relationship     0.780
marital-status     0.770
    occupation     0.730
     education     0.716
 education-num     0.716
           age     0.682
hours-per-week     0.672
           sex     0.618
  capital-gain     0.590
     workclass     0.585
          race     0.538
  capital-loss     0.534
native-country     0.529
        fnlwgt     0.504


In [29]:
print(target01.groupby(df["race"]).agg(["mean", "size"]).round(3))

                     mean   size
race                            
Amer-Indian-Eskimo  0.117    470
Asian-Pac-Islander  0.269   1519
Black               0.121   4685
Other               0.123    406
White               0.254  41762


## Stage 6
- S6-1  I split BEFORE any step that learns from data, such as imputation ,
        standard scaler and target encoding .
- S6-2  stratify=y keeps the same class ratio in train and test.
- S6-3  An 80/20 split of 48,790 cleaned rows gives train = 39032 and test = 9758 .
- S6-4  DummyClassifier(strategy="most_frequent") always predicts the most common class.
    Its AUC is 0.5 , which means "no better than a coin".
- S6-5  The test set is opened 1 time(s) in the whole project.
- S6-6  With 5-fold CV on the training part, I do NOT need a separate
    validation set, because every training row takes a turn as validation.
- S6-7  get_dummies before the split is acceptable because it uses no
    target information, but StandardScaler before the split is NOT because
    it learns the mean and std of the data. (Explain the difference
    in your own words.)


In [68]:
df = df.drop_duplicates()
print(df.shape)        # must show (48790, 15)

(48790, 15)


In [69]:
y = (df["income"] == ">50K").astype(int)

In [70]:
drop_cols =["sex", "education", "fnlwgt", "race", "income"]
X = df.drop(columns=drop_cols)
print(X.shape)

(48790, 10)


In [71]:
X["native-country"] = X["native-country"].where(X["native-country"] == "United-States", "Other")

In [72]:
X = pd.get_dummies(X, dtype=int)

In [73]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [74]:
print(X_train.shape, X_test.shape)
print(y_train.mean().round(4), y_test.mean().round(4))

(39032, 44) (9758, 44)
0.2394 0.2394


In [75]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
proba = dummy.predict_proba(X_test)[:, 1]
print("AUC", round(roc_auc_score(y_test, proba), 4),
      round(accuracy_score(y_test, dummy.predict(X_test)), 4))

AUC 0.5 0.7606


## Stage 7
- S7-1  Category text -> numbers with 0/1 (one-hot). A column with 4
        categories becomes 4 new 0/1 columns.
- S7-2  Columns I drop: education (reason: redundant, duplicate of education-num), fnlwgt (reason: nearly zero correlation with the target column ), and the target itself.
- S7-3  Number of columns after encoding (read from your shape): 44 .
    Rows per column in the training part: 39032 ÷ 44 = 887.1 .
- S7-4  Numeric imputer strategy: "median" . For categories, I keep "?" as its
    own category (so no imputer is needed for them).
- S7-5  Logistic regression needs scaling because it uses gradient descent optimization and penalty ; the random
    forest does not because trees only ask "is x > threshold?".
- S7-6  The Pipeline order is: Imputer -> Scaler -> Model .
- S7-7  Inside cross-validation, the scaler is re-fitted on each training part,
    which stops data leakage from validation rows.

In [76]:
print(X_train.shape[1])
print(round(X_train.shape[0] / X_train.shape[1],  1))

44
887.1


In [77]:
print(X_train.isna().sum().sum())
print(round(X_train.dtypes.value_counts()))

0
int64    44
Name: count, dtype: int64


In [78]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

lr_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

rf_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
])

In [79]:
lr_pipe.fit(X_train.head(500), y_train.head(500))
print(lr_pipe.predict_proba(X_train.head(3)).shape)

(3, 2)


In [80]:
rf_pipe.fit(X_train.head(500), y_train.head(500))
print(rf_pipe.predict_proba(X_train.head(3)).shape)

(3, 2)


## Stage 8

- S8-1  Confusion matrix: rows = true class, columns = predicted class.
- S8-2  Precision = of the people I SAID earn >50K, how many actually do.
        Recall    = of the people who REALLY earn >50K, how many I found.
- S8-3  If a model has 80 % accuracy and the baseline has 76 %, it adds only
        4 points. Is that a lot? No. Accuracy can fool me here because the classes are imbalanced
        (only about 24 % earn >50K): a useless model that always guesses "<=50K" scores 76 %
        without finding a single high earner. So I also read recall, precision and AUC.
- S8-4  Train AUC 0.99 and validation AUC 0.89 -> overfitting . Both 0.60 -> underfitting .
- S8-5  A default random forest (no depth limit) has a big gap because every
        tree can memorise the training rows.
- S8-6  Fill from YOUR results:
        model                 train AUC   validation AUC   gap
        logistic regression     0.9048          0.9039     0.0008
        forest (default)        0.9967          0.8936     0.1031
- S8-7  My reading of the gap for each model:
    * Logistic regression: train 0.9048 vs validation 0.9039, a gap of only 0.0008. No overfitting,
      and both scores are far above the 0.5 baseline, so it generalises well.
    * Default forest: train 0.9967 vs validation 0.8936, a gap of 0.1031. It overfits: with no depth
      limit every tree memorises the training rows. Its validation AUC is even lower than the simple
      logistic regression (0.8936 vs 0.9039), so being more flexible did not help.
- S8-8  At the default 0.5 threshold, the logistic regression finds 59.3 % of true >50K
        people (recall). To find more of them I can lower the threshold (0.4 gives recall 0.687,
        0.3 gives 0.783). The cost is more false alarms: precision falls from 0.731 to 0.676 at 0.4
        and to 0.610 at 0.3, because the model now says "high earner" about more people who are not.


In [83]:
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [85]:
models = [("logistic regression", lr_pipe),
          ("forest (default)", rf_pipe)]
rows = []
for name, pipe in models:
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring="roc_auc", return_train_score=True, n_jobs=1)
    train_auc = res["train_score"].mean()
    val_auc = res["test_score"].mean()
    rows.append((name, round(train_auc, 4), round(val_auc, 4), round(train_auc - val_auc, 4)))

print(pd.DataFrame(rows, columns=["model", "train AUC", "validation AUC", "gap"]).to_string(index=False))

              model  train AUC  validation AUC    gap
logistic regression     0.9048          0.9039 0.0008
   forest (default)     0.9967          0.8936 0.1031


In [87]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
oof = {}
for name, pipe in models:
    oof[name] = cross_val_predict(pipe, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
    pred = (oof[name] >= 0.5).astype(int)
    print("\n", name, "accuracy", round(accuracy_score(y_train, pred), 4 ))
    print(confusion_matrix(y_train, pred))
    print(classification_report(y_train, pred, digits=3))


 logistic regression accuracy 0.8503
[[27648  2039]
 [ 3805  5540]]
              precision    recall  f1-score   support

           0      0.879     0.931     0.904     29687
           1      0.731     0.593     0.655      9345

    accuracy                          0.850     39032
   macro avg      0.805     0.762     0.780     39032
weighted avg      0.844     0.850     0.845     39032


 forest (default) accuracy 0.8465
[[27306  2381]
 [ 3609  5736]]
              precision    recall  f1-score   support

           0      0.883     0.920     0.901     29687
           1      0.707     0.614     0.657      9345

    accuracy                          0.847     39032
   macro avg      0.795     0.767     0.779     39032
weighted avg      0.841     0.847     0.843     39032



In [89]:
from sklearn.metrics import precision_score, recall_score

for name in oof:
    print(name)
    for  t in [0.5, 0.4, 0.3]:
        p = (oof[name] >= t).astype(int)
        print("  Threshold", t, "precision", round(precision_score(y_train, p), 3),
              "recall", round(recall_score(y_train, p), 3))

logistic regression
  Threshold 0.5 precision 0.731 recall 0.593
  Threshold 0.4 precision 0.676 recall 0.687
  Threshold 0.3 precision 0.61 recall 0.783
forest (default)
  Threshold 0.5 precision 0.707 recall 0.614
  Threshold 0.4 precision 0.653 recall 0.688
  Threshold 0.3 precision 0.604 recall 0.758


### Stage 8 - Four label
QUESTION : Which of the two first models generalises better, and what does the train-vs-validation gap say?
CHECK    : cross_validate(..., return_train_score=True), then cross_val_predict with confusion_matrix and classification_report.
FINDING  : Logistic regression train 0.9048 vs validation 0.9039 (gap 0.0008); default forest 0.9967 vs 0.8936 (gap 0.1031). At threshold 0.5 the logistic regression finds only 59.3 % of the >50K people.
DECISION : The default forest overfits, so in Stage 9 I limit its depth and leaf size; and because recall is weak I will also look at a lower threshold.

## Stage 9

- S9-1  CV gives mean ± std . If two models differ by less than that std,
        I call it a tie and pick the simpler one.
- S9-2  A search over 4 settings with 6 values each, 5 folds, tries
        6 x 6 x 6 x 6 x 5 = 6,480 fits. That is why I would use
        RandomizedSearchCV instead of Grid.
- S9-3  Both searches use X_train only. They never see the test set.
- S9-4  After the search, best_estimator_ has already been refit on all of the
        training rows.
- S9-5  For a random forest, the settings most connected to overfitting are
        max_depth and min_samples_leaf (Day 10).
- S9-6  Fill from YOUR results (mean ± std):
            logistic regression       0.9039 ± 0.0048
            forest default            0.8936 ± 0.0042
            forest after search       0.9179 ± 0.0043   (grid best: max_depth None, min_samples_leaf 5;
                                                         the random search found 0.918, a tie)
- S9-7  Of the top 5 settings from my search, the spread of scores is 0.0047 ,
        compared with a std of 0.0043 , so they are essentially tied (the first four are
        within one std of the best). I pick the simpler of the tied settings, not "the winner".
- S9-8  The biggest gain came from moving away from max_depth=None with min_samples_leaf=1 (the default
        setting): 0.8936 -> 0.9179, about +0.024, not from fine-tuning the last decimal.
- S9-9  Optional: if feature selection (SelectKBest) did not beat the full set,
        I would still prefer fewer columns only if the score stays within the std of the full set
        (simpler, faster, easier to explain). With logistic regression k=30 (0.9032) is about as good as
        all 44 columns (0.9039), but it adds a pipeline step for no gain, so I keep all 44.
- Capital-gain test (Cell 8): without capital-gain the logistic regression falls from 0.9039 to 0.8863
  and the tuned forest from 0.9179 to 0.8970. Both drops (about 0.018 and 0.021) are several times the
  std (about 0.004), so the model leans heavily on capital-gain.


In [ ]:
from sklearn.model_selection import cross_val_score, GridSearchCV, RandomizedSearchCV

def cv_report(pipe, cols=None):
    Xt = X_train if cols is None else X_train[cols]
    s = cross_val_score(pipe, Xt, y_train, cv=cv, scoring="roc_auc", n_jobs=1)
    return round(float(s.mean()), 4), round(float(s.std()), 4)

In [ ]:
print("logistic regression", cv_report(lr_pipe))
print("forest default     ", cv_report(rf_pipe))

In [92]:
param_grid = {"model__max_depth": [6, 10, 14, None],
              "model__min_samples_leaf": [1, 5, 20]}
print(4 * 3 * 5, "fits")

grid = GridSearchCV(rf_pipe, param_grid, cv=cv, scoring="roc_auc", n_jobs=1)
grid.fit(X_train, y_train)
print(grid.best_params_, round(grid.best_score_, 4))

60 fits
{'model__max_depth': None, 'model__min_samples_leaf': 5} 0.9179


In [93]:
res = pd.DataFrame(grid.cv_results_).sort_values("rank_test_score")
show = ["param_model__max_depth", "param_model__min_samples_leaf",
        "mean_test_score", "std_test_score"]

top = res.head(5)[show]
print(top.round(4).to_string(index=False))
print("spread of top 5: ", round(top["mean_test_score"].max() - top["mean_test_score"].min(), 4))
print(res.tail(1)[show].round(4).to_string(index=False))

param_model__max_depth  param_model__min_samples_leaf  mean_test_score  std_test_score
                  None                              5           0.9179          0.0043
                    14                              1           0.9171          0.0043
                    14                              5           0.9163          0.0044
                  None                             20           0.9144          0.0047
                    14                             20           0.9132          0.0047
spread of top 5:  0.0047
param_model__max_depth  param_model__min_samples_leaf  mean_test_score  std_test_score
                  None                              1           0.8936          0.0042


In [ ]:
print("forest after search", cv_report(grid.best_estimator_))

In [95]:
dist = {"model__max_depth": [6, 8, 10, 12, 14, None],
        "model__min_samples_leaf": [1, 3, 5, 10, 20, 40],
        "model__max_features": ["sqrt", 0.3, 0.5]}
rnd = RandomizedSearchCV(rf_pipe, dist, n_iter=10, cv=cv, scoring="roc_auc", random_state=42, n_jobs=1)
rnd.fit(X_train, y_train)
print(rnd.best_params_, round(rnd.best_score_, 4))

{'model__min_samples_leaf': 3, 'model__max_features': 0.5, 'model__max_depth': 12} 0.918


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

for k in [10, 20, 30, 44]:
    sel_pipe = Pipeline([
        ("impupter", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("select", SelectKBest(score_func=f_classif, k=k)),
        ("model", LogisticRegression(max_iter=2000))
    ])
    print("k =", k, cv_report(sel_pipe))

In [ ]:
no_gain = [c for c in X_train.columns if c != "capital-gain"]

print("logistic regression  with / without:",
      cv_report(lr_pipe), cv_report(lr_pipe, cols=no_gain))
print("tuned forest         with / without:",
      cv_report(grid.best_estimator_), cv_report(grid.best_estimator_, cols=no_gain))

### Stage 9 - Four label
QUESTION : Does tuning beat the simple model, and are the top settings really different?
CHECK    : cross_val_score (cv_report), GridSearchCV, RandomizedSearchCV, SelectKBest inside a Pipeline, and a with/without test for capital-gain.
FINDING  : The forest goes from 0.8936 (default) to 0.9179 ± 0.0043 (tuned), against 0.9039 ± 0.0048 for logistic regression; the top 5 grid settings span 0.0047, about one std. Without capital-gain the tuned forest falls to 0.8970.
DECISION : Choose the tuned forest (its lead of 0.014 over logistic regression is bigger than the std), keep all 44 columns, keep capital-gain but report my dependence on it as a limit.

## Stage 10

- S10-1 The test set is opened 1 time(s).
- S10-2 If the test AUC is close to the CV AUC (within about one std ), the
        estimate is honest. If it is far lower, I first suspect leakage .
        (Mine: test 0.9225 vs CV 0.9179 ± 0.0043, a gap of 0.0046, about one std. Honest.)
- S10-3 Permutation importance asks: how much does the score drop when one
        column is shuffled?
- S10-4 Top 3 columns by importance (your output): capital-gain (0.0445) , education-num (0.0265) ,
        marital-status_Married-civ-spouse (0.0234) .
        Does "relationship" or "marital-status" ranking surprise you? Why?
        Married-civ-spouse is 3rd but relationship_Husband is only 7th (0.0052). The two columns overlap,
        so when one is shuffled the model still uses the other and the credit is split. It is not a
        surprise that marital status matters (44.6 % of married-civ-spouse earn >50K vs 4.6 % of
        never-married), but that does not mean marriage causes a higher income.
- S10-5 Fairness: I removed sex in the model. Reason: so the model cannot use it directly.
        Even so, the model can still treat men and women differently because
        relationship and marital-status still encode it (Husband is 99.99 % male), and the groups really
        differ in the data. On the test set the model finds 52.0 % of high-earning women versus 65.3 % of
        high-earning men, and it predicts >50K for 7.1 % of women versus 25.3 % of men
        (actual: 11.5 % vs 30.4 %).
- S10-6 Fill the final paragraph:
    "On the Adult Census data, a tuned random forest (max_depth None, min_samples_leaf 5, 200 trees)
     reached a test AUC of 0.9225 against a baseline of 0.50. It finds 63.1 % of high earners
     (recall) with 79.0 % precision at the default 0.5 threshold. The main drivers are capital-gain,
     education-num and marital status (Married-civ-spouse). Limits: the model leans heavily on
     capital-gain, which may be part of how the income label was built (without it the CV AUC drops
     from 0.918 to 0.897); it misses 37 % of high earners and finds fewer high-earning women (52.0 %)
     than men (65.3 %) even though sex was removed; the data is an old census extract and the test set
     was opened once. Next step: lower the threshold using out-of-fold predictions to raise recall and
     check the cost in precision, look at the group gap, and read how capital-gain and the income label
     were defined."


In [99]:
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, classification_report

final_model = grid.best_estimator_
final_model.fit(X_train, y_train)

proba = final_model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print("test AUC", round(roc_auc_score(y_test, proba), 4),
      "accuracy", round(accuracy_score(y_test, pred), 4))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, digits=3))

test AUC 0.9225 accuracy 0.8714
[[7029  393]
 [ 862 1474]]
              precision    recall  f1-score   support

           0      0.891     0.947     0.918      7422
           1      0.790     0.631     0.701      2336

    accuracy                          0.871      9758
   macro avg      0.840     0.789     0.810      9758
weighted avg      0.867     0.871     0.866      9758



In [ ]:
print("CV AUC mean, std", cv_report(final_model))

In [101]:
from sklearn.inspection import permutation_importance

pi = permutation_importance(final_model, X_test, y_test, scoring="roc_auc",
                            n_repeats=5, random_state=42, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=X_test.columns).sort_values(ascending=False)
print(imp.head(10).round(4))

capital-gain                         0.0445
education-num                        0.0265
marital-status_Married-civ-spouse    0.0234
age                                  0.0192
hours-per-week                       0.0093
capital-loss                         0.0085
relationship_Husband                 0.0052
occupation_Exec-managerial           0.0044
marital-status_Never-married         0.0035
occupation_Prof-specialty            0.0024
dtype: float64


In [102]:
test_info = df.loc[X_test.index, ["sex", "race"]].assign(true=y_test, pred=pred)

print(test_info.groupby("sex").agg(share_predicted_high=("pred", "mean"),
                                   share_actually_high=("true", "mean"),
                                   n=("pred", "size")).round(3))

print(test_info[test_info["true"] == 1].groupby("sex")["pred"].mean().round(3))

        share_predicted_high  share_actually_high     n
sex                                                    
Female                 0.071                0.115  3326
Male                   0.253                0.304  6432
sex
Female    0.520
Male      0.653
Name: pred, dtype: float64


### Stage 10 - Four label
QUESTION : How good is the chosen model on unseen data, what drives it, and does it treat groups differently?
CHECK    : opened X_test once (roc_auc_score, confusion_matrix, classification_report), permutation_importance, and recall by sex.
FINDING  : Test AUC 0.9225 vs CV 0.9179 ± 0.0043 (gap about one std, honest); recall 63.1 % with 79.0 % precision; top drivers capital-gain 0.0445, education-num 0.0265, Married-civ-spouse 0.0234; recall is 52.0 % for women vs 65.3 % for men.
DECISION : Report the model with its limits (capital-gain dependence, weak recall, group gap), do not tune again on the test set, and next lower the threshold using out-of-fold predictions and study the group gap.